# DATA304 · ModernBERT Cross-Encoder · 랭킹, 리뷰 속성, 리뷰–상품 일치 학습

ColBERT epoch 5가 고른 상품 100개를 재정렬하는 **ModernBERT 한 모델**을 Gold TRAIN에서 3개 에포크 학습한다. 검색기는 고정하고, ModernBERT 백본과 새 랭킹 헤드·속성 헤드 6개를 함께 갱신한다. 하나의 리뷰 묶음과 각 후보 상품이 일치하는지를 기존 랭킹 점수로도 학습한다. 시작 가중치는 공개 `answerdotai/ModernBERT-base`다.

SageMaker의 기존 PyTorch 커널에서 위에서 아래로 **Run All** 한다. 마지막 셀은 후보 준비와 길이 진단을 마친 뒤 학습을 시작한다. 모든 실험 결과는 `data304_outputs/modernbert_cross_multiaspect/` 아래에 쓴다. 파일과 모델 설정을 읽기 위한 접근 이외에 외부 서비스로 데이터를 보내는 단계는 없다.

기본 모델의 설정 및 출력 사양은 공식 자료에서 확인할 수 있다. [ModernBERT 모델 카드](https://huggingface.co/answerdotai/ModernBERT-base) · [Transformers ModernBERT 문서](https://huggingface.co/docs/transformers/en/model_doc/modernbert)

## 1. 실행 환경과 학습 설정

PyTorch는 SageMaker 커널의 CUDA 빌드를 사용한다. Transformers가 없거나 ModernBERT 도입 버전보다 오래된 경우에만 4.57.6을 설치한다. BF16을 지원하는 장치에서는 autocast를 사용하며, 그 외에는 FP32로 계산한다. 기존 후보 캐시를 모두 읽을 수 있으면 FAISS 로딩도 생략한다.

In [ ]:
from collections import Counter, defaultdict
from contextlib import nullcontext
from datetime import datetime
from zipfile import BadZipFile
from functools import lru_cache
from importlib.metadata import PackageNotFoundError, version
from itertools import chain, islice
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import hashlib
import importlib
import importlib.util
import json
import os
import math
import re
import random
import subprocess
import sys

# SageMaker의 CUDA용 PyTorch를 사용한다. 이 셀은 현재 커널에 필요한 Python 패키지만 보충한다.
import torch
from packaging.version import Version


def ensure_python_package(module, requirement):
    """현재 커널에서 import할 수 없는 패키지만 설치한다."""
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', requirement])
        importlib.invalidate_caches()


try:
    has_modernbert = Version(version('transformers')) >= Version('4.48.0')
except PackageNotFoundError:
    has_modernbert = False
if not has_modernbert:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers==4.57.6'])
    importlib.invalidate_caches()
ensure_python_package('pandas', 'pandas')
import numpy as np
import pandas as pd
import transformers
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from transformers import AutoConfig, AutoModel, AutoTokenizer

if 'modernbert' not in transformers.models.auto.configuration_auto.CONFIG_MAPPING:
    raise RuntimeError(
        f'현재 커널의 transformers {transformers.__version__}가 ModernBERT를 지원하지 않습니다.'
    )

# SageMaker 기본 폴더를 우선 사용한다. 다른 위치에서는 프로젝트 루트에서 커널을 시작한다.
sage_project = Path('/home/ec2-user/SageMaker/20262R0136DATA30400')
PROJECT_DIR = sage_project if sage_project.is_dir() else Path.cwd()
RUN_DIR = PROJECT_DIR / 'data304_outputs'
OUTPUT_DIR = RUN_DIR / 'modernbert_cross_multiaspect'
CROSS_BACKBONE = 'answerdotai/ModernBERT-base'
CROSS_MAX_LENGTH = 1024
COLBERT_EPOCH = 5
ANN_NPROBE = 32
ANN_PER_TOKEN = 1024
ANN_POOL = 900
CANDIDATE_TOPK = 100
VAL_LIMIT = 350
EPOCHS = 3
SEED = 42
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
# 순위를 주목표로 두고, 속성 예측과 리뷰–상품 일치 판별을 보조한다.
RANKING_LOSS_WEIGHT = 1.0
ASPECT_LOSS_WEIGHT = 0.1
PAIR_LOSS_WEIGHT = 0.1  # 0.0으로 바꾸면 업데이트에서 이진 분류 항을 끌 수 있다.
ASPECT_KEYS = ('category', 'brand', 'color', 'material', 'size', 'features')
HARD_COUNT, ORIGINAL_COUNT = 4, 4
NEGATIVE_COUNT = HARD_COUNT + ORIGINAL_COUNT
ORIGINAL_MODE = 'mixed'
# 한 번에 최대 2개 쿼리를 처리하고 총 4개 쿼리의 gradient를 모아 AdamW를 갱신한다.
MICRO_BATCH_QUERIES, GRAD_ACCUM_STEPS = 2, 2
PAIR_MICRO_BATCH, EVAL_PAIR_BATCH = 3, 8
EFFECTIVE_QUERIES = MICRO_BATCH_QUERIES * GRAD_ACCUM_STEPS
BM25_TOPK = 100
K1, B, EPSILON = 1.5, 0.75, 0.25
MINING_CHUNK_SIZE = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
print(
    '프로젝트',
    PROJECT_DIR,
    '| PyTorch',
    torch.__version__,
    '| transformers',
    transformers.__version__,
    '| 장치',
    DEVICE,
    '| BF16',
    USE_BF16,
    flush=True,
)

## 2. 입력 파일과 고정 검색기 확인

기존 config에서 ColBERT의 백본·토큰 길이·인덱스 구성을 복원한다. `split.json`의 TRAIN 순서와 앞 350개 VAL 쿼리를 유지한다. `ann_nprobe=32`, 토큰당 이웃 1,024개, ANN 후보 900개, MaxSim 후보 100개가 이번 실행값이다. 캐시 키에는 검색기 가중치와 카탈로그·Gold·split 파일의 SHA-256이 들어간다.

In [ ]:
def read_json(path):
    # 기존 JSON 파일의 UTF-8 BOM 유무를 모두 처리한다.
    return json.loads(path.read_text(encoding='utf-8-sig'))


def write_json(path, value):
    # 완성된 설정을 나중에 점수표와 대조할 수 있도록 사람이 읽을 수 있는 형식으로 쓴다.
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


@lru_cache(maxsize=32)
def file_hash(path, size, modified_ns):
    # 모델과 입력이 같을 때만 캐시를 공유한다. 같은 실행에서 큰 파일을 매번 다시 읽지 않는다.
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def sha256(path):
    # 파일을 교체했으면 크기 또는 수정 시각이 바뀌므로 새 해시를 얻는다.
    stat = path.stat()
    return file_hash(path, stat.st_size, stat.st_mtime_ns)


def signature(value):
    # 파일 내용과 설정이 일치하는 후보 캐시·실험을 찾기 위한 해시를 만든다.
    data = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(data).hexdigest()


def load_pt(path):
    # CPU 메모리 매핑으로 저장한 모델과 AdamW에서 필요한 텐서부터 읽는다.
    return torch.load(path, map_location='cpu', mmap=True, weights_only=True)


def amp_context():
    # 원래 실행처럼 지원되는 CUDA에서는 bfloat16으로 순전파한다.
    return torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()


def metadata_text(item):
    # 학습 때 사용한 네 필드의 이름, 순서, [SEP]를 그대로 지켜 임베딩 입력을 맞춘다.
    fields = [
        ('title', 'Product'),
        ('store', 'Store'),
        ('features', 'Features'),
        ('description', 'Description'),
    ]
    parts = [f'{label}: {item[key].strip()}' for key, label in fields if item[key].strip()]
    return ' [SEP] '.join(parts) or 'Product metadata not available'


def gold_query(row):
    # 리뷰 다섯 개는 원래 순서를 지킨다. Gold 카테고리 경로는 기존 방식의 오답을 뽑을 때 사용한다.
    if len(row['reviews']) != 5:
        raise ValueError(f'리뷰가 5개가 아닌 쿼리 {row["query_id"]}')
    return {
        'category_path': tuple(
            level.strip() for level in (row.get('category_path') or '').split('>') if level.strip()
        ),
        'query_id': row['query_id'],
        'item_id': row['item_id'],
        'reviews': [review['text'].strip() for review in row['reviews']],
    }


CATALOG_PATH = PROJECT_DIR / 'catalog.json'
GOLD_PATH = PROJECT_DIR / 'train_gold.json'
SPLIT_PATH = RUN_DIR / 'split.json'
VOCAB_PATH = RUN_DIR / 'aspect_vocabulary_and_config.json'
ASPECT_PATH = RUN_DIR / 'metadata_aspects.json'
RETRIEVER_DIR = RUN_DIR / 'additional_training' / f'epoch_{COLBERT_EPOCH:02d}'
RETRIEVER_WEIGHTS = RETRIEVER_DIR / 'checkpoints' / 'colbert_gold.pt'
missing = [
    str(p)
    for p in (CATALOG_PATH, GOLD_PATH, SPLIT_PATH, VOCAB_PATH, ASPECT_PATH, RETRIEVER_WEIGHTS)
    if not p.is_file()
]
if missing:
    raise FileNotFoundError('필요한 입력 파일:\n' + '\n'.join(missing))
if min(MICRO_BATCH_QUERIES, GRAD_ACCUM_STEPS, PAIR_MICRO_BATCH, EVAL_PAIR_BATCH) < 1:
    raise ValueError('모든 배치 관련 설정은 양수여야 합니다.')

# 이전 실행의 어휘 번호 및 ColBERT 설정을 읽는다. 이번 검색의 nprobe는 명시한 32를 적용한다.
vocab_record = read_json(VOCAB_PATH)
VOCABS = {k: vocab_record['vocabularies'][k] for k in ASPECT_KEYS}
for key, vocab in VOCABS.items():
    if not vocab or set(vocab.values()) != set(range(len(vocab))):
        raise ValueError(f'{key} vocabulary의 출력 열 번호를 확인하세요.')
CFG = dict(vocab_record['config'])
manifest = RUN_DIR / 'run_manifest.json'
if manifest.is_file():
    CFG.update(read_json(manifest)['config'])
CFG['ann_nprobe'] = ANN_NPROBE
catalog = read_json(CATALOG_PATH)
ITEM_IDS = [entry['item_id'] for entry in catalog]
ITEM_TO_IDX = {iid: i for i, iid in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(entry) for entry in catalog]
del catalog
if len(ITEM_TO_IDX) != len(ITEM_IDS) or len(ITEM_IDS) < CANDIDATE_TOPK:
    raise ValueError('catalog.json의 고유 item_id 개수 또는 Top-100 후보 수를 확인하세요.')
gold = read_json(GOLD_PATH)
BY_QUERY = {r['query_id']: gold_query(r) for r in gold}
if len(BY_QUERY) != len(gold):
    raise ValueError('train_gold.json에 중복 query_id가 있습니다.')
del gold
split = read_json(SPLIT_PATH)
train_ids, val_ids = split['train_query_ids'], split['val_query_ids']
if (
    split['seed'] != SEED
    or not train_ids
    or len(val_ids) < VAL_LIMIT
    or len(set(train_ids + val_ids)) != len(train_ids) + len(val_ids)
    or not set(train_ids + val_ids) <= BY_QUERY.keys()
):
    raise ValueError('split.json의 seed, query_id, TRAIN·VAL 수를 확인하세요.')
TRAIN = [BY_QUERY[q] for q in train_ids]
full_val = [BY_QUERY[q] for q in val_ids]
if {r['item_id'] for r in TRAIN} & {r['item_id'] for r in full_val} or any(
    r['item_id'] not in ITEM_TO_IDX for r in chain(TRAIN, full_val)
):
    raise ValueError('Gold 상품이 TRAIN·VAL에 겹치거나 카탈로그에 없습니다.')
VAL = full_val[:VAL_LIMIT]
del full_val
TRAIN_CATEGORY = {ITEM_TO_IDX[r['item_id']]: r['category_path'] for r in TRAIN if r['category_path']}
CAT_TO_DOCS, PARENT_TO_DOCS = defaultdict(list), defaultdict(list)
for doc, path in TRAIN_CATEGORY.items():
    CAT_TO_DOCS[path].append(doc)
    PARENT_TO_DOCS[path[:-1]].append(doc)

# 이전 cross 후보 폴더를 만든 코드와 동일한 서명이다. 파일 경로가 같아도 내용이나 nprobe가 다르면 키가 바뀐다.
INDEX_SPEC = {
    'weights': sha256(RETRIEVER_WEIGHTS),
    'catalog': sha256(CATALOG_PATH),
    'seed': SEED,
    'precision': 'bfloat16' if USE_BF16 else 'float32',
    'config': {
        k: CFG[k]
        for k in (
            'backbone',
            'dimension',
            'metadata_tokens',
            'ann_nlist',
            'index_doc_batch',
            'index_add_docs',
        )
    },
}
MINING_SPEC = {
    'version': 1,
    'index': INDEX_SPEC,
    'original_index_files': None,
    'gold': sha256(GOLD_PATH),
    'split': sha256(SPLIT_PATH),
    'query_tokens': CFG['query_tokens'],
    'ann_nprobe': ANN_NPROBE,
    'ann_per_token': ANN_PER_TOKEN,
    'ann_pool': ANN_POOL,
    'candidate_topk': CANDIDATE_TOPK,
}
old_mixed = RUN_DIR / 'cross_hard4_mixed4_lr5e-6'
mining_key, index_key = signature(MINING_SPEC)[:20], signature(INDEX_SPEC)[:20]
CACHE_ROOTS = [
    p / 'candidate_cache' / mining_key for p in (OUTPUT_DIR, old_mixed, RUN_DIR / 'cross_hard_negative')
]
INDEX_PATHS = [p / 'indexes' / index_key for p in (RETRIEVER_DIR, old_mixed)]
TOPK = CANDIDATE_TOPK
HARD_SPEC = {
    'version': 1,
    'train_mining': MINING_SPEC,
    'bm25_topk': BM25_TOPK,
    'k1': K1,
    'b': B,
    'epsilon': EPSILON,
    'lexical_query': 'all_five_raw_lower_alnum_tf',
    'corpus': 'CAT_TEXTS',
    'hard_count': HARD_COUNT,
    'order': 'stable_union_colbert_then_bm25_exact_maxsim',
    'dtype': 'fp64_bm25',
}
print(
    f'TRAIN {len(TRAIN):,} | VAL {len(VAL):,} | 카탈로그 {len(ITEM_IDS):,} | epoch {COLBERT_EPOCH} 후보 키 {mining_key}'
)
print('고정 ColBERT', RETRIEVER_WEIGHTS, '\n기존 후보 캐시', *CACHE_ROOTS[1:], sep='\n', flush=True)

## 3. TRAIN의 BM25·ColBERT 후보와 Exact MaxSim 오답 선정

예전 두 Cross 실험과 이번 실험의 후보 캐시에서 같은 검색 키·분할·쿼리 순서의 파일을 찾는다. 누락된 묶음만 저장된 epoch 5 인덱스와 동결한 ColBERT로 검색한다. 후보와 일치하는 기존 인덱스가 모두 없으면 필요한 경로가 표시된다.

TRAIN은 리뷰 원문 다섯 개 전체로 BM25 Top-100을 검색한다. ColBERT Top-100과의 중복을 제거한 최대 200개에서 Gold를 제외한 뒤, 모두 고정 ColBERT의 **Exact MaxSim**으로 다시 채점해 상위 오답 4개를 고른다. BM25는 후보를 제안하고, 순위는 MaxSim 점수로 결정한다. 계산한 오답·점수·양쪽 원래 순위는 새 `train_hard_cache/`에 50쿼리씩 저장한다. 후보 ID만 남은 첫 실행은 epoch 5 상품 토큰·마스크가 필요하다. VAL은 기존 ColBERT Top-100을 그대로 사용한다.

각 쿼리의 정답을 첫 열에 놓고 검색 오답 4개, 전체 카탈로그 무작위 2개, TRAIN 상품 중 같은 부모·다른 말단 1개, 같은 말단 1개를 붙인다. 정답·중복은 제외하고 부족한 카테고리 자리는 무작위로 채운다. TRAIN의 카테고리 후보군에는 TRAIN 레이블만 들어간다.

In [ ]:
class ColBERTRetriever(nn.Module):
    def __init__(self, config, dimension):
        super().__init__()
        self.query_encoder = AutoModel.from_config(config)
        self.doc_encoder = AutoModel.from_config(config)
        self.query_proj = nn.Linear(config.hidden_size, dimension, bias=False)
        self.doc_proj = nn.Linear(config.hidden_size, dimension, bias=False)

    # 학습 노트북과 같은 정규화와 유효 토큰 마스크를 사용해 MaxSim용 벡터를 반환한다.
    def encode(self, tokens, side):
        backbone = self.query_encoder if side == 'query' else self.doc_encoder
        projection = self.query_proj if side == 'query' else self.doc_proj
        hidden = backbone(**tokens).last_hidden_state
        mask = tokens['attention_mask'].bool()
        special = torch.zeros_like(mask)
        for token_id in SPECIAL_IDS:
            special |= tokens['input_ids'].eq(token_id)
        valid = mask & ~special
        empty = ~valid.any(dim=-1)
        if empty.any():
            valid[empty] = mask[empty]
        return F.normalize(projection(hidden), p=2, dim=-1), valid


def tokenize(texts, max_length):
    out = COL_TOKENIZER(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    return {k: v.to(DEVICE) for k, v in out.items()}


def open_token_index(folder):
    # 같은 인덱스의 네 파일을 읽고 상품 수·토큰 길이·임베딩 차원이 맞는지 확인한다.
    import faiss

    emb = np.load(folder / 'doc_embeddings.npy', mmap_mode='r')
    mask = np.load(folder / 'doc_masks.npy', mmap_mode='r')
    token2doc = np.load(folder / 'token_to_item.npy', mmap_mode='r')
    index = faiss.read_index(str(folder / 'token_ivf.index'))
    expected = (len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension'])
    if (
        emb.shape != expected
        or mask.shape != expected[:2]
        or mask.dtype != np.bool_
        or index.d != expected[2]
        or token2doc.ndim != 1
        or index.ntotal != len(token2doc)
    ):
        raise ValueError(f'상품 인덱스의 크기가 현재 설정과 맞지 않습니다. {folder}')
    if not len(token2doc) or token2doc.min() < 0 or token2doc.max() >= len(ITEM_IDS):
        raise ValueError(f'인덱스의 토큰-상품 대응표를 확인하세요. {folder}')
    if index.nlist < ANN_NPROBE:
        raise ValueError('기존 인덱스의 IVF 셀 수가 지정한 nprobe보다 작습니다.')
    index.nprobe = ANN_NPROBE
    return {'index': index, 'token2doc': token2doc, 'emb': emb, 'mask': mask}


@torch.inference_mode()
def encode_five_reviews(model, reviews):
    tokens = tokenize(reviews, CFG['query_tokens'])
    with amp_context():
        embeddings, valid = model.encode(tokens, 'query')
    return embeddings.float().cpu().numpy(), valid.cpu().numpy()


@torch.inference_mode()
def maxsim_scores(query, query_mask, index_bundle, doc_indices):
    # 후보의 유효 상품 토큰 전체를 읽고, 리뷰 토큰마다 가장 높은 내적을 더한다.
    # 최종 점수 하나는 다섯 리뷰의 MaxSim 점수 평균이다.
    qe = torch.as_tensor(query, device=DEVICE)
    qm = torch.as_tensor(query_mask, device=DEVICE)
    scores = []
    for start in range(0, len(doc_indices), 96):
        rows = doc_indices[start : start + 96]
        de = torch.as_tensor(np.asarray(index_bundle['emb'][rows]).astype(np.float32), device=DEVICE)
        dm = torch.as_tensor(np.asarray(index_bundle['mask'][rows]), device=DEVICE)
        with amp_context():
            simil = torch.einsum('rqd,cmd->rcqm', qe, de)
            simil = simil.masked_fill(~dm[None, :, None, :], -1e4)
            review_scores = (simil.max(-1).values * qm[:, None, :]).sum(-1)
        scores.append(review_scores.mean(0).float().cpu().numpy())
    return np.concatenate(scores)


def scored_pool(ann_order, pool_size, topk):
    # ANN 적중 여부는 보충 전 순위로 센다. 실제 후보 부족 시에만 원래 검색기와 같은 순서로 보충한다.
    pool = ann_order[:pool_size]
    if len(pool) < topk:
        seen = set(pool)
        need = min(pool_size, len(ITEM_IDS)) - len(pool)
        pool.extend(islice((i for i in range(len(ITEM_IDS)) if i not in seen), need))
    return pool


def ann_order(query, mask, bundle):
    # 같은 (리뷰 토큰, 상품)에서 최대 유사도 하나를 고른 뒤 양수 부분을 상품별로 합산한다.
    # 정렬·집계를 NumPy로 처리해 TRAIN 전체에서 토큰 이웃을 파이썬 반복문으로 순회하는 비용을 줄인다.
    tokens = np.ascontiguousarray(query[mask], dtype=np.float32)
    distances, hits = bundle['index'].search(tokens, ANN_PER_TOKEN)
    rows, columns = np.nonzero(hits >= 0)
    if rows.size == 0:
        return []
    docs = bundle['token2doc'][hits[rows, columns]]
    keys = rows.astype(np.int64) * len(ITEM_IDS) + docs
    order = np.argsort(keys, kind='stable')
    keys = keys[order]
    starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1]
    maxima = np.maximum.reduceat(distances[rows, columns][order], starts)
    docs = keys[starts] % len(ITEM_IDS)
    rough = np.bincount(docs, weights=np.maximum(maxima, 0.0), minlength=len(ITEM_IDS))
    touched = np.unique(docs)
    return touched[np.argsort(-rough[touched], kind='stable')[:ANN_POOL]].tolist()


@torch.inference_mode()
def retrieve_candidates(row, retriever, bundle):
    # ANN 후보를 exact MaxSim으로 다시 정렬한다. 원본처럼 top-K보다 적게 찾은 때만 풀을 보충한다.
    query, mask = encode_five_reviews(retriever, row['reviews'])
    ann = ann_order(query, mask, bundle)
    gold = ITEM_TO_IDX[row['item_id']]
    ann_rank = ann.index(gold) + 1 if gold in ann else 0
    pool = scored_pool(ann, ANN_POOL, TOPK)
    scores = maxsim_scores(query, mask, bundle, pool)
    indices = np.argsort(-scores, kind='stable')[:TOPK]
    return np.asarray(pool, dtype=np.int32)[indices], ann_rank

In [ ]:
def open_saved_retriever(need_ann=True):
    """캐시에 없는 쿼리만 검색한다. 인덱스는 디스크에서 읽고 epoch 5 모델의 gradient를 끈다."""
    global COL_TOKENIZER, SPECIAL_IDS
    required = ['doc_embeddings.npy', 'doc_masks.npy']
    if need_ann:
        required += ['token_ivf.index', 'token_to_item.npy']
    folder = next((p for p in INDEX_PATHS if all((p / k).is_file() for k in required)), None)
    if folder is None:
        raise FileNotFoundError(
            '저장된 epoch 5 검색 파일이 필요합니다: '
            + ', '.join(required)
            + '\n'
            + '\n'.join(map(str, INDEX_PATHS))
        )
    info = folder / 'index_info.json'
    if info.is_file() and read_json(info) != INDEX_SPEC:
        raise ValueError(f'저장된 인덱스 가중치·카탈로그·인코딩 설정이 다릅니다. {info}')
    # 캐시만으로 충분할 때는 FAISS를 설치하거나 ColBERT를 GPU에 올릴 필요가 없다.
    if need_ann:
        if importlib.util.find_spec('faiss') is None:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', f'numpy=={np.__version__}', 'faiss-cpu']
            )
            importlib.invalidate_caches()
        import faiss

        faiss.omp_set_num_threads(min(8, os.cpu_count() or 4))
        bundle = open_token_index(folder)
    else:
        # 기존 ANN 후보가 있으면 BM25 후보 재채점에는 동일한 상품 토큰·마스크 두 파일만 필요하다.
        emb = np.load(folder / 'doc_embeddings.npy', mmap_mode='r')
        mask = np.load(folder / 'doc_masks.npy', mmap_mode='r')
        if (
            emb.shape != (len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension'])
            or mask.shape != emb.shape[:2]
            or mask.dtype != np.bool_
        ):
            raise ValueError(f'epoch 5 상품 토큰과 마스크의 차원을 확인하세요. {folder}')
        bundle = {'emb': emb, 'mask': mask}
    COL_TOKENIZER = AutoTokenizer.from_pretrained(CFG['backbone'], use_fast=True)
    SPECIAL_IDS = set(COL_TOKENIZER.all_special_ids)
    model = ColBERTRetriever(AutoConfig.from_pretrained(CFG['backbone']), CFG['dimension'])
    weights = load_pt(RETRIEVER_WEIGHTS)
    model.load_state_dict({k: weights[k] for k in model.state_dict()}, strict=True)
    return model.to(DEVICE).eval().requires_grad_(False), bundle


def load_shard(path, qids):
    """같은 검색 키 안에서도 쿼리 순서·배열 크기·상품 범위·ANN 순위를 확인한다."""
    with np.load(path, allow_pickle=False) as saved:
        docs, ranks = saved['candidates'], saved['ann_ranks']
        if (
            not np.array_equal(saved['query_ids'], qids)
            or docs.shape != (len(qids), CANDIDATE_TOPK)
            or ranks.shape != (len(qids),)
            or docs.dtype.kind not in 'iu'
            or ranks.dtype.kind not in 'iu'
            or docs.min() < 0
            or docs.max() >= len(ITEM_IDS)
            or np.any(np.diff(np.sort(docs, axis=1), axis=1) == 0)
            or ranks.min() < 0
            or ranks.max() > ANN_POOL
        ):
            raise ValueError('후보 캐시의 query_id, 후보 목록 또는 ANN 순위가 다릅니다.')
        return docs.astype(np.int32, copy=False), ranks.astype(np.int32, copy=False)


def candidate_tables(log):
    """TRAIN·VAL 별로 이전 실험의 50쿼리 파일을 읽고, 부족한 묶음만 새 출력 폴더에 완성해 둔다."""
    model = index = None
    tables = {}
    for name, rows in [('train', TRAIN), ('val', VAL)]:
        qids = [r['query_id'] for r in rows]
        key = f'{name}_{signature(qids)[:16]}'
        candidates = np.empty((len(rows), TOPK), dtype=np.int32)
        ann_ranks = np.empty(len(rows), dtype=np.int32)
        reused = 0
        log_line(log, f'{name} 후보 준비 | {len(rows)} queries | {mining_key}', True)
        for start in range(0, len(rows), MINING_CHUNK_SIZE):
            end = min(start + MINING_CHUNK_SIZE, len(rows))
            shard = Path(key) / f'{start:06d}_{end:06d}.npz'
            expected_ids = np.asarray(qids[start:end], dtype=str)
            for cache in CACHE_ROOTS:
                path = cache / shard
                if not path.is_file():
                    continue
                try:
                    docs, ranks = load_shard(path, expected_ids)
                except (OSError, ValueError, KeyError, EOFError, BadZipFile) as error:
                    log_line(log, f'캐시 건너뜀 | {path} | {error}')
                    continue
                candidates[start:end], ann_ranks[start:end] = docs, ranks
                reused += end - start
                break
            else:
                if model is None:
                    model, index = open_saved_retriever()
                for i in range(start, end):
                    candidates[i], ann_ranks[i] = retrieve_candidates(rows[i], model, index)
                target = CACHE_ROOTS[0] / shard
                target.parent.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(dir=target.parent) as tmp:
                    staged = Path(tmp) / target.name
                    np.savez(
                        staged,
                        query_ids=expected_ids,
                        candidates=candidates[start:end],
                        ann_ranks=ann_ranks[start:end],
                    )
                    staged.replace(target)
                log_line(log, f'mining {name} | {end}/{len(rows)} queries saved')
        tables[name] = candidates, ann_ranks
        log_line(log, f'{name} 후보 완료 | 캐시 {reused:,}건 | 새 검색 {len(rows) - reused:,}건', True)
    return tables


def sample_original_negatives(gold, hard, category, rng):
    # 기존 Gold sampler의 추출 순서를 유지하되, 검색 오답 4개도 처음부터 제외한다.
    picked = []
    excluded = {gold, *hard}

    def fill_random(target):
        # 전체 카탈로그에서 현재 목록이 target 길이가 될 때까지 중복 없이 뽑는다.
        while len(picked) < target:
            doc = rng.randrange(len(ITEM_IDS))
            if doc not in excluded:
                picked.append(doc)
                excluded.add(doc)

    def add_category(pool, count):
        # 정답과 앞서 뽑은 오답을 제외한 뒤, 카테고리에 남아 있는 만큼만 뽑는다.
        available = [doc for doc in pool if doc not in excluded]
        chosen = rng.sample(available, min(count, len(available)))
        picked.extend(chosen)
        excluded.update(chosen)

    if ORIGINAL_MODE == 'mixed' and category:
        random_count = ORIGINAL_COUNT // 2
        sibling_count = (ORIGINAL_COUNT - random_count) // 2
        fill_random(random_count)
        siblings = (doc for doc in PARENT_TO_DOCS.get(category[:-1], ()) if TRAIN_CATEGORY[doc] != category)
        add_category(siblings, sibling_count)
        add_category(CAT_TO_DOCS.get(category, ()), ORIGINAL_COUNT - random_count - sibling_count)
    fill_random(ORIGINAL_COUNT)
    return picked


def mixed_groups(rows, candidates, rng):
    # 행마다 [Gold, MaxSim 상위 오답 4개, 기존 방식 오답 4개]를 만들어 Cross의 정답 열을 0번으로 맞춘다.
    groups = np.empty((len(rows), 1 + NEGATIVE_COUNT), dtype=np.int32)
    for i, row in enumerate(rows):
        gold = ITEM_TO_IDX[row['item_id']]
        ranked = candidates[i]
        hard = ranked[ranked != gold][:HARD_COUNT].tolist()
        category = TRAIN_CATEGORY.get(gold, row['category_path'])
        original = sample_original_negatives(gold, hard, category, rng)
        groups[i] = [gold, *hard, *original]
    return groups

In [ ]:
def tokenize_lexical(text):
    # 프로젝트 BM25의 소문자 영숫자 토큰화를 그대로 사용한다.
    return re.findall(r'[a-z0-9]+', text.lower())


def build_bm25(texts):
    # CSC의 열 하나가 한 단어를 포함한 상품 목록이다.
    ensure_python_package('sklearn', 'scikit-learn')
    from sklearn.feature_extraction import DictVectorizer

    vec = DictVectorizer(sort=False, dtype=np.float64)
    counts = (Counter(tokenize_lexical(text)) for text in texts)
    index = vec.fit_transform(counts).tocsc()
    lengths = np.asarray(index.sum(axis=1)).ravel()
    df = np.diff(index.indptr).astype(np.int64)
    n_docs = index.shape[0]
    idf = np.empty(len(df), dtype=np.float64)
    idf_sum = 0.0
    # 원래 BM25Okapi가 카탈로그에서 처음 본 단어 순서대로 IDF를 더하는 방식과 맞춘다.
    for column, freq in enumerate(df.tolist()):
        value = math.log(n_docs - freq + 0.5) - math.log(freq + 0.5)
        idf[column] = value
        idf_sum += value
    idf[idf < 0] = EPSILON * (idf_sum / len(idf))
    length_norm = K1 * (1 - B + B * lengths / lengths.mean())
    tf = index.data
    index.data = np.repeat(idf, df) * (tf * (K1 + 1) / (tf + length_norm[index.indices]))
    return vec.vocabulary_, index


def lexical_scores(text, vocabulary, index):
    """리뷰 5개 원문 전체의 단어 빈도로 BM25 점수를 구한다. 반복 단어는 postings를 한 번만 읽는다."""
    scores = np.zeros(index.shape[0], dtype=np.float64)
    for term, count in Counter(tokenize_lexical(text)).items():
        column = vocabulary.get(term)
        if column is not None:
            start, end = index.indptr[column : column + 2]
            scores[index.indices[start:end]] += count * index.data[start:end]
    return scores


@torch.inference_mode()
def union_hard_negatives(row, colbert, lexical_index, retriever, vectors):
    """두 Top-100의 합집합을 동일한 Exact MaxSim 척도로 평가하고 정답을 제외한 상위 4개를 고른다."""
    bm_scores = lexical_scores(' '.join(row['reviews']), *lexical_index)
    lexical = np.argsort(-bm_scores, kind='stable')[:BM25_TOPK].tolist()
    colbert = colbert.tolist()
    union = list(dict.fromkeys(colbert + lexical))
    gold = ITEM_TO_IDX[row['item_id']]
    pool = [doc for doc in union if doc != gold]
    query, mask = encode_five_reviews(retriever, row['reviews'])
    scores = maxsim_scores(query, mask, vectors, pool)
    if not np.isfinite(scores).all():
        raise ValueError(f'Exact MaxSim 점수를 확인하세요. {row["query_id"]}')
    best = np.argsort(-scores, kind='stable')[:HARD_COUNT]
    docs = [pool[j] for j in best]
    # 순위가 0이면 해당 검색기의 원래 후보에 없었다. 두 검색기에서 같이 나온 상품도 한 번만 센다.
    col_rank = {doc: j for j, doc in enumerate(colbert, 1)}
    bm_rank = {doc: j for j, doc in enumerate(lexical, 1)}
    return docs, scores[best], [col_rank.get(d, 0) for d in docs], [bm_rank.get(d, 0) for d in docs]


def train_hard_candidates(rows, colbert_candidates, log):
    """TRAIN 전용 오답을 한 번 계산해 저장한다. VAL 검색 테이블과 이전 후보 캐시는 수정하지 않는다."""
    hard = np.empty((len(rows), HARD_COUNT), dtype=np.int32)
    folder = (
        OUTPUT_DIR
        / 'train_hard_cache'
        / signature(HARD_SPEC)[:20]
        / signature([r['query_id'] for r in rows])[:16]
    )
    folder.mkdir(parents=True, exist_ok=True)
    lexical_index = retriever = vectors = None
    reused = 0
    bm_only = bm_queries = 0
    for start in range(0, len(rows), MINING_CHUNK_SIZE):
        end = min(start + MINING_CHUNK_SIZE, len(rows))
        qids = np.asarray([r['query_id'] for r in rows[start:end]], dtype=str)
        gold = np.asarray([ITEM_TO_IDX[r['item_id']] for r in rows[start:end]])
        col = colbert_candidates[start:end]
        pool_hash = hashlib.sha256(np.ascontiguousarray(col, dtype=np.int32).tobytes()).hexdigest()
        # ColBERT 후보 행의 내용까지 달라지면 캐시 파일 이름이 달라진다.
        path = folder / f'{start:06d}_{end:06d}_{pool_hash[:16]}.npz'
        cached = False
        if path.is_file():
            try:
                with np.load(path, allow_pickle=False) as z:
                    docs, scores = z['hard_docs'], z['hard_maxsim']
                    cpos, bpos = z['colbert_ranks'], z['bm25_ranks']
                    shape = (len(qids), HARD_COUNT)
                    cached = (
                        np.array_equal(qids, z['query_ids'])
                        and str(z['colbert_hash']) == pool_hash
                        and all(a.shape == shape for a in (docs, scores, cpos, bpos))
                        and all(a.dtype.kind in 'iu' for a in (docs, cpos, bpos))
                        and (docs >= 0).all()
                        and (docs < len(ITEM_IDS)).all()
                        and not (docs == gold[:, None]).any()
                        and not (np.diff(np.sort(docs, axis=1), axis=1) == 0).any()
                        and np.isfinite(scores).all()
                        and (np.diff(scores, axis=1) <= 0).all()
                        and ((cpos >= 0) & (cpos <= CANDIDATE_TOPK) & (bpos >= 0) & (bpos <= BM25_TOPK)).all()
                        and ((cpos > 0) | (bpos > 0)).all()
                    )
            except (OSError, ValueError, KeyError, EOFError, BadZipFile):
                cached = False
        if cached:
            reused += len(qids)
        else:
            if retriever is None:
                log_line(log, 'TRAIN BM25 Top-100 ∪ ColBERT Top-100 | epoch 5 Exact MaxSim 재채점 시작', True)
                retriever, vectors = open_saved_retriever(need_ann=False)
                lexical_index = build_bm25(CAT_TEXTS)
            values = [
                union_hard_negatives(row, col[j], lexical_index, retriever, vectors)
                for j, row in enumerate(rows[start:end])
            ]
            docs, scores, cpos, bpos = map(np.asarray, zip(*values))
            with TemporaryDirectory(dir=folder) as tmp:
                staged = Path(tmp) / path.name
                np.savez(
                    staged,
                    query_ids=qids,
                    colbert_hash=pool_hash,
                    hard_docs=docs.astype(np.int32),
                    hard_maxsim=scores,
                    colbert_ranks=cpos,
                    bm25_ranks=bpos,
                )
                staged.replace(path)
            log_line(log, f'BM25+MaxSim TRAIN | {end}/{len(rows)} queries saved', end % 500 == 0)
        hard[start:end] = docs
        selected_bm_only = (cpos == 0) & (bpos > 0)
        bm_only += int(selected_bm_only.sum())
        bm_queries += int(selected_bm_only.any(axis=1).sum())
    log_line(
        log,
        f'혼합 검색 오답 준비 | 캐시 {reused:,}/{len(rows):,}쿼리 | '
        f'BM25에서만 찾은 상품이 선택된 횟수 {bm_only:,}/{hard.size:,}, 쿼리 {bm_queries:,}개',
        True,
    )
    return hard

## 4. Gold 리뷰의 속성 타깃

카테고리는 Gold 상품에 연결된 경로를 기존 vocabulary 번호로 변환한다. TRAIN에는 TRAIN 경로만 쓰며, VAL 쿼리의 경로는 검증 타깃을 만드는 데에만 쓴다. 나머지 속성은 저장된 메타데이터 값이 리뷰 중 하나에 문자 그대로 등장해야 양성 타깃이 된다. 미등록 값과 미확인 행은 제외한다.

Features는 근거가 있는 행만 다중 레이블 BCE에 참여한다. 리뷰와 메타데이터 양쪽에서 확인한 feature는 1, 한쪽에만 있는 값은 무시한다. 그 행의 나머지 어휘는 원래 multi-hot 규칙처럼 0이다. `aspect_coverage.csv`는 분할별로 각 속성에 실제 감독을 제공한 쿼리 수를 담는다.

In [ ]:
def parse_aspects(x):
    # 저장된 메타데이터의 표기와 대소문자를 원래 vocabulary 규칙에 맞춘다.
    d = {}
    for key in ASPECT_KEYS:
        val = x.get(key, [] if key == 'features' else 'unknown')
        if key == 'features':
            if isinstance(val, str):
                val = [] if val.lower() == 'unknown' else [val]
            if not isinstance(val, list):
                val = []
            d[key] = [
                str(y).strip().lower()
                for y in val[:5]
                if str(y).strip().lower() not in ('', 'unknown', 'none')
            ]
        else:
            d[key] = str(val).strip().lower() if val is not None else 'unknown'
            if not d[key] or d[key] in ('none', 'n/a', 'null'):
                d[key] = 'unknown'
    return d


# 카테고리 풀은 TRAIN에서만 만든다. VAL의 Gold 경로는 그 쿼리 자체의 평가 타깃에만 사용한다.
unknown_values = {'', 'unknown', 'none', 'null', 'n/a', 'nan'}


def review_targets(row, annotated):
    """Gold 경로와 리뷰 원문의 리터럴 근거를 저장한 어휘의 열 번호로 바꾼다."""
    reviews = [r.lower() for r in row['reviews']]
    path = ' > '.join(row['category_path']).lower()
    result = {k: [] for k in ASPECT_KEYS}
    if path in VOCABS['category'] and all(
        level.lower() not in unknown_values for level in row['category_path']
    ):
        result['category'] = [VOCABS['category'][path]]
    labels = parse_aspects(annotated)

    def mentioned(value):
        # 원래 review-aspect 규칙과 같은 소문자 부분 문자열 검사다. 리뷰 경계를 가로질러 매칭하지 않는다.
        return len(value) >= 2 and any(value in text for text in reviews)

    for key in ('brand', 'color', 'material', 'size'):
        value = labels[key]
        if value not in unknown_values and value in VOCABS[key] and mentioned(value):
            result[key] = [VOCABS[key][value]]
    # 원본의 multi-hot BCE를 사용한다. 1은 metadata와 리뷰 양쪽에서 확인한 feature다.
    # 둘 중 한쪽에서만 확인한 값은 불확실하므로 0으로 강제하지 않고 열 단위로 마스킹한다.
    vocab = VOCABS['features']
    known = {vocab[v] for v in labels['features'] if v in vocab and v not in unknown_values and len(v) >= 2}
    visible = {i for v, i in vocab.items() if v not in unknown_values and mentioned(v)} if known else set()
    result['features'] = sorted(known & visible)
    result['feature_ignore'] = sorted(known ^ visible)
    return result


def init_targets():
    """저장된 metadata 속성만 사용한다. 누락된 상품은 알고 있는 Gold 카테고리만 감독할 수 있다."""
    aspects = read_json(ASPECT_PATH)
    if not isinstance(aspects, dict) or not aspects.keys() & ITEM_TO_IDX.keys():
        raise ValueError('metadata_aspects.json의 item_id 대응을 확인하세요.')
    targets = {}
    coverage = []
    for split_name, rows in [('TRAIN', TRAIN), ('VAL', VAL)]:
        for row in rows:
            record = aspects.get(row['item_id']) or {}
            targets[row['query_id']] = review_targets(row, record)
        coverage.append(
            {
                'split': split_name,
                'queries': len(rows),
                **{key: sum(bool(targets[row['query_id']][key]) for row in rows) for key in ASPECT_KEYS},
            }
        )
    if not any(coverage[0][k] for k in ASPECT_KEYS):
        raise ValueError(
            'TRAIN에 유효한 aspect 타깃이 하나도 없습니다. vocabulary와 데이터 대응을 확인하세요.'
        )
    return targets, pd.DataFrame(coverage)


def epoch_groups(number, candidates):
    """원래 Cross 셔플과 sampler의 난수 순서를 재현한다. 반환 배열의 행은 섞인 학습 순서다."""
    rng = random.Random(SEED + number - 1 + 233)
    order = list(range(len(TRAIN)))
    rng.shuffle(order)
    rows = [TRAIN[i] for i in order]
    groups = mixed_groups(rows, candidates[order], rng)
    return rows, groups

## 5. ModernBERT 입력과 길이 진단

순서대로 연결한 리뷰 다섯 개 사이에 native `[SEP]` ID가 들어간다. 리뷰마다 토큰을 제한하지 않는다. 순위 경로는 리뷰 묶음과 metadata를 tokenizer의 두 시퀀스로 전달하고, 리뷰 전용 경로는 첫 번째 시퀀스만 사용한다. 두 경로 모두 마지막에 한 번 1,024 토큰 제한을 적용한다. 쌍에서는 `longest_first`로 긴 쪽부터 줄인다.

첫 optimizer step 전에 **TRAIN epoch 1의 실제 9개 후보 쌍 전체**, **VAL의 실제 상위 100개 쌍 전체**, 각 분할의 **리뷰 단독 입력**을 집계한다. 내부·외부 구분자를 포함하고 패딩은 제외한다. 초과 비율은 0~1이며, 잘린 비율은 전체 토큰을 분모로 한 값과 입력별 비율의 평균을 모두 제공한다.

In [ ]:
# 학습 가능한 모델과 토크나이저는 같은 ModernBERT 리비전이다. 처음 한 번 기본 파일을 받아 로컬에서 입력을 처리한다.
MODERN_CONFIG = AutoConfig.from_pretrained(CROSS_BACKBONE)
REVISION = getattr(MODERN_CONFIG, '_commit_hash', None)
CROSS_TOKENIZER = AutoTokenizer.from_pretrained(CROSS_BACKBONE, revision=REVISION, use_fast=True)
if (
    MODERN_CONFIG.model_type != 'modernbert'
    or not CROSS_TOKENIZER.is_fast
    or CROSS_MAX_LENGTH > MODERN_CONFIG.max_position_embeddings
    or any(
        token is None
        for token in (
            CROSS_TOKENIZER.pad_token_id,
            CROSS_TOKENIZER.cls_token_id,
            CROSS_TOKENIZER.sep_token_id,
        )
    )
):
    raise ValueError('ModernBERT 기본 토크나이저, special token 또는 최대 길이를 확인하세요.')


# [CLS]·[SEP]는 체크포인트에 등록된 토큰을 쓴다. pair template가 없는 tokenizer 배포본도 동일하게 처리한다.
def check_pair_template(tokenizer):
    """한 쌍이 [CLS] A [SEP] B [SEP]가 되는지 작은 예시로 검사한다."""
    a, b = 'review', 'product'
    aa, bb = [tokenizer(t, add_special_tokens=False)['input_ids'] for t in (a, b)]
    cls, sep = tokenizer.cls_token_id, tokenizer.sep_token_id
    if tokenizer(a)['input_ids'] != [cls, *aa, sep] or tokenizer(a, b)['input_ids'] != [
        cls,
        *aa,
        sep,
        *bb,
        sep,
    ]:
        from tokenizers.processors import TemplateProcessing

        start, end = tokenizer.cls_token, tokenizer.sep_token
        tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
            single=f'{start} $A {end}',
            pair=f'{start} $A {end} $B {end}',
            special_tokens=[(start, cls), (end, sep)],
        )
    if tokenizer(a)['input_ids'] != [cls, *aa, sep] or tokenizer(a, b)['input_ids'] != [
        cls,
        *aa,
        sep,
        *bb,
        sep,
    ]:
        raise ValueError('ModernBERT pair에 두 입력의 경계 토큰을 적용할 수 없습니다.')


check_pair_template(CROSS_TOKENIZER)
CROSS_TOKENIZER.truncation_side = 'right'
CROSS_TOKENIZER.padding_side = 'right'
# 리뷰별 토큰 제한 없이 5개를 순서대로 연결한다. 내부 구분자는 4개이며 바깥 토큰은 tokenizer가 붙인다.
REVIEW_TEXTS = {r['query_id']: f' {CROSS_TOKENIZER.sep_token} '.join(r['reviews']) for r in chain(TRAIN, VAL)}


def modern_inputs(review_texts, metadata=None):
    """학습·평가에 공통인 유일한 ModernBERT 인코딩 경로. metadata가 없으면 리뷰 단독 시퀀스다."""
    encoded = CROSS_TOKENIZER(
        review_texts,
        text_pair=metadata,
        add_special_tokens=True,
        padding=True,
        truncation='longest_first',
        max_length=CROSS_MAX_LENGTH,
        return_token_type_ids=False,
        return_tensors='pt',
    )
    return encoded['input_ids'].to(DEVICE), encoded['attention_mask'].to(DEVICE)


def raw_lengths(texts):
    """특수 토큰 추가·잘림·패딩 전의 토큰 수를 구한다. 같은 상품은 한 번만 토큰화한다."""
    lengths = []
    for start in range(0, len(texts), 128):
        encoded = CROSS_TOKENIZER(
            texts[start : start + 128],
            add_special_tokens=False,
            truncation=False,
            padding=False,
            return_length=True,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )
        lengths.extend(encoded['length'])
    return np.asarray(lengths, dtype=np.int64)


def length_diagnostics(train_rows, train_groups, val_candidates):
    """실제 epoch 1 학습 쌍과 VAL Top-100 전체의 길이를 학습 전에 측정한다. 패딩은 분모에서 뺀다."""
    meta = raw_lengths(CAT_TEXTS)
    single = CROSS_TOKENIZER.num_special_tokens_to_add(pair=False)
    pair = CROSS_TOKENIZER.num_special_tokens_to_add(pair=True)
    summaries = []
    for name, rows, docs in [('train_epoch1', train_rows, train_groups), ('val', VAL, val_candidates)]:
        reviews = raw_lengths([REVIEW_TEXTS[r['query_id']] for r in rows])
        for view, lens in [
            ('ranking', (reviews[:, None] + meta[docs] + pair).ravel()),
            ('reviews_only', reviews + single),
        ]:
            lost = np.maximum(lens - CROSS_MAX_LENGTH, 0)
            summaries.append(
                {
                    'split': name,
                    'input': view,
                    'n': len(lens),
                    'mean': lens.mean(),
                    'median': np.median(lens),
                    **{f'p{p}': np.percentile(lens, p) for p in (90, 95, 99)},
                    **{f'fraction_over_{k}': (lens > k).mean() for k in (512, 1024, 2048)},
                    'mean_tokens_lost_at_1024': lost.mean(),
                    'fraction_all_tokens_lost_at_1024': lost.sum() / lens.sum(),
                    'mean_fraction_lost_per_input': (lost / lens).mean(),
                }
            )
    return pd.DataFrame(summaries)

## 6. 하나의 ModernBERT에서 계산하는 세 가지 손실

**랭킹과 리뷰–상품 일치 판별**은 같은 관련도 점수를 쓴다. 쿼리 하나는 원래 순서대로 묶은 리뷰 5개다. 이 묶음에 정답 상품과 오답 8개의 메타데이터를 각각 붙여 점수 9개를 얻는다.

| 손실 | 입력과 정답 | 기본 가중치 |
|---|---|---:|
| Ranking | 쿼리별 점수 9개에 listwise cross-entropy를 적용한다. 정답 인덱스는 0이다. | 1.0 |
| Aspect | 리뷰만 별도로 인코딩한다. 근거가 있는 속성에 단일 분류 CE 또는 다중 레이블 BCE를 적용하고, 관측된 속성끼리 평균한다. | 0.1 |
| Pair | 같은 점수 9개에 BCEWithLogits를 적용한다. 리뷰 묶음과 연결된 정답 상품은 1, 오답 상품 8개는 각각 0이다. | 0.1 |

Pair loss는 쿼리마다 **정답 BCE의 절반 + 오답 8개 평균 BCE의 절반**을 사용한다. 따라서 오답 수가 많아도 두 클래스의 비중은 같다. 랭킹 CE가 후보 사이의 순서를 학습하는 동안, Pair BCE는 같은 점수에서 정답을 양의 방향으로, 오답을 음의 방향으로 민다. 배치를 여러 조각으로 처리해도 쿼리별 9개 점수를 모은 뒤 두 손실을 계산한다.

In [ ]:
class ModernCross(nn.Module):
    """두 입력 경로가 하나의 ModernBERT를 공유한다. 순위·속성 헤드는 독립적인 선형층이다."""

    def __init__(self, backbone):
        super().__init__()
        self.encoder = backbone
        dim = backbone.config.hidden_size
        self.ranking_head = nn.Linear(dim, 1)
        self.aspect_heads = nn.ModuleDict({k: nn.Linear(dim, len(VOCABS[k])) for k in ASPECT_KEYS})

    def pooled(self, ids, mask):
        # ModernBERT-base의 평균 풀링 설정에 맞춰 padding 위치를 뺀 마지막 은닉 상태를 평균한다.
        hidden = self.encoder(input_ids=ids, attention_mask=mask, return_dict=True).last_hidden_state
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weights).sum(dim=1, dtype=torch.float32) / mask.sum(dim=1, keepdim=True).clamp_min(1)

    def rank(self, ids, mask):
        # 리뷰와 상품을 함께 본 점수 하나가 listwise 랭킹과 일치 여부 판별에 모두 쓰인다.
        return self.ranking_head(self.pooled(ids, mask)).flatten()


def new_model(resuming):
    """첫 에포크는 공개 ModernBERT-base 가중치로 시작한다. 재개 시에는 이 실험의 전체 state_dict를 복원한다."""
    args = dict(config=MODERN_CONFIG, attn_implementation='sdpa')
    # HF 4.x의 선택적 torch.compile을 끄고, SageMaker에서 별도 FlashAttention 설치 없이 실행한다.
    if hasattr(MODERN_CONFIG, 'reference_compile'):
        MODERN_CONFIG.reference_compile = False
    if resuming:
        backbone = AutoModel.from_config(MODERN_CONFIG, attn_implementation='sdpa')
    else:
        backbone = AutoModel.from_pretrained(CROSS_BACKBONE, revision=REVISION, **args)
    backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    return ModernCross(backbone).to(DEVICE)


def rank_scores(model, queries, docs, chunk):
    """후보를 잘게 나눠도 같은 listwise 분모를 유지하기 위해 모든 scalar logit을 다시 연결한다."""
    scores = []
    for start in range(0, len(docs), chunk):
        ids, mask = modern_inputs(
            queries[start : start + chunk], [CAT_TEXTS[i] for i in docs[start : start + chunk]]
        )
        with amp_context():
            if model.training and len(docs) > chunk:
                # 여러 후보의 중간 activation을 계속 보관하지 않고 역전파 때 동일한 dropout 상태로 재계산한다.
                out = checkpoint(model.rank, ids, mask, use_reentrant=False, preserve_rng_state=True)
            else:
                out = model.rank(ids, mask)
        scores.append(out.float())
    return torch.cat(scores)


def aspect_counts(targets):
    """유효한 (쿼리, 속성) 개수를 세어 micro-batch 분할 전후의 손실 가중치를 맞춘다."""
    return np.asarray([sum(bool(row[k]) for row in targets) for k in ASPECT_KEYS], dtype=np.int64)


def aspect_loss_sums(model, texts, targets):
    """리뷰 단독 forward를 한 번 수행해 유효한 행의 단일분류 CE·feature 다중분류 BCE 합을 구한다."""
    ids, mask = modern_inputs(texts)
    with amp_context():
        pooled = model.pooled(ids, mask)
        losses = []
        for key, head in model.aspect_heads.items():
            valid = [i for i, t in enumerate(targets) if t[key]]
            if not valid:
                losses.append(pooled.new_zeros(()))
                continue
            logits = head(pooled[valid]).float()
            if key == 'features':
                labels = torch.zeros_like(logits)
                known = torch.ones_like(logits, dtype=torch.bool)
                for j, i in enumerate(valid):
                    labels[j, targets[i][key]] = 1.0
                    known[j, targets[i]['feature_ignore']] = False
                element = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
                loss_sum = ((element * known).sum(1) / known.sum(1).clamp_min(1)).sum()
            else:
                labels = torch.tensor([targets[i][key][0] for i in valid], device=DEVICE, dtype=torch.long)
                loss_sum = F.cross_entropy(logits, labels, reduction='sum')
            losses.append(loss_sum)
    return torch.stack(losses)


def pair_loss_per_query(logits):
    """[쿼리 수, 후보 수] 점수에서 정답 한 쌍과 오답 쌍들이 각각 절반을 차지하는 BCE를 구한다."""
    # mixed_groups가 각 행의 첫 열에 그 리뷰 묶음의 Gold item_id에 해당하는 상품 인덱스를 놓는다. 나머지는 서로 다른 오답이다.
    scores = logits.float()
    targets = torch.zeros_like(scores)
    targets[:, 0] = 1.0
    per_pair = F.binary_cross_entropy_with_logits(scores, targets, reduction='none')
    return 0.5 * (per_pair[:, 0] + per_pair[:, 1:].mean(dim=1))


def loss_report(rank_sum, pair_sum, n, aspect_sums, counts):
    """랭킹·Pair는 쿼리 평균, 속성은 관측된 레이블 기준 평균을 기록하고 설정 가중치로 합산한다."""
    active = counts > 0
    aspect = float(np.mean(aspect_sums[active] / counts[active])) if active.any() else 0.0
    rank = float(rank_sum / n)
    pair = float(pair_sum / n)
    return {
        'ranking_loss': rank,
        'aspect_loss': aspect,
        'pair_loss': pair,
        'total_loss': RANKING_LOSS_WEIGHT * rank + ASPECT_LOSS_WEIGHT * aspect + PAIR_LOSS_WEIGHT * pair,
    }

## 7. 학습과 Top-100 검증

기본 설정은 AdamW 한 번당 총 4쿼리다. 한 번에 2쿼리, 순위 forward당 최대 3쌍을 처리한다. 순·역전파 중 CUDA OOM이면 현재 스텝을 같은 난수 상태에서 재시도하며 쿼리 수, 이어서 동시 처리할 쌍 수를 줄인다. 입력 길이와 한 업데이트의 전체 쿼리 수는 유지한다.

VAL의 ranking·pair loss는 매번 같은 정답 1개와 오답 8개로 계산한다. 속성 loss는 해당 쿼리의 리뷰 5개로 계산한다. 실제 검색 밖의 정답이나 mixed 오답은 loss 계산용으로만 채점한다. Precision@1, Recall@10, MRR@100은 실제 상위 100개에서 재정렬한 결과로 계산하며, 정답이 후보에서 빠졌으면 0이다. 조건부 Top-1만 정답이 후보에 든 쿼리를 분모로 쓴다.

In [ ]:
def torch_rng():
    """Dropout의 CPU·GPU 난수 상태를 저장해 OOM 재시도와 epoch 재개 시 같은 순서로 시작한다."""
    return {
        'cpu': torch.get_rng_state(),
        'gpu': torch.cuda.get_rng_state(DEVICE) if DEVICE.type == 'cuda' else None,
    }


def restore_rng(saved):
    """체크포인트에 들어 있는 난수 상태를 현재 장치에 복원한다."""
    torch.set_rng_state(saved['cpu'])
    if DEVICE.type == 'cuda' and saved['gpu'] is not None:
        torch.cuda.set_rng_state(saved['gpu'], DEVICE)


def backward_window(model, rows, groups, micro, pair_batch):
    """랭킹·Pair는 전체 쿼리 수로, 속성은 전체 유효 레이블 수로 정규화해 gradient를 누적한다."""
    targets = [TARGETS[r['query_id']] for r in rows]
    counts = aspect_counts(targets)
    # 동일한 유효 배치에서 micro 크기를 바꿔도 각 관측 레이블에 같은 가중치가 들어간다.
    coeffs = np.divide(1.0, counts, out=np.zeros(len(counts)), where=counts > 0) / max(
        int((counts > 0).sum()), 1
    )
    coeffs = torch.as_tensor(coeffs, dtype=torch.float32, device=DEVICE)
    sums = torch.zeros(2 + len(ASPECT_KEYS), device=DEVICE)  # [랭킹 합, Pair 합, 속성별 합...]
    for start in range(0, len(rows), micro):
        stop = min(start + micro, len(rows))
        texts = [REVIEW_TEXTS[r['query_id']] for r in rows[start:stop]]
        docs = groups[start:stop]
        repeated = [text for text in texts for _ in range(docs.shape[1])]
        logits = rank_scores(model, repeated, docs.ravel(), pair_batch).reshape(len(texts), docs.shape[1])
        gold_col = torch.zeros(len(texts), device=DEVICE, dtype=torch.long)
        rank_sum = F.cross_entropy(logits, gold_col, reduction='sum')
        pair_sum = pair_loss_per_query(logits).sum()
        attr = aspect_loss_sums(model, texts, targets[start:stop])
        query_loss_sum = RANKING_LOSS_WEIGHT * rank_sum + PAIR_LOSS_WEIGHT * pair_sum
        objective = query_loss_sum / len(rows) + ASPECT_LOSS_WEIGHT * torch.dot(attr, coeffs)
        objective.backward()
        sums += torch.cat([rank_sum.detach().view(1), pair_sum.detach().view(1), attr.detach()])
    return sums.cpu().numpy().astype(np.float64), counts


def train_epoch(model, optimizer, epoch, candidates, log, sizes):
    """매 epoch에 mixed 오답을 다시 뽑고 가중치 갱신별 ranking·aspect·pair·total loss를 로그에 남긴다."""
    rows, groups = epoch_groups(epoch, candidates)
    model.train()
    sums = np.zeros(2 + len(ASPECT_KEYS))
    counts = np.zeros(len(ASPECT_KEYS), dtype=np.int64)
    starts = range(0, len(rows), EFFECTIVE_QUERIES)
    for step, start in enumerate(starts, 1):
        stop = min(start + EFFECTIVE_QUERIES, len(rows))
        rng = torch_rng()
        while True:
            optimizer.zero_grad(set_to_none=True)
            try:
                values, seen = backward_window(
                    model, rows[start:stop], groups[start:stop], sizes['queries'], sizes['pairs']
                )
                break
            except torch.cuda.OutOfMemoryError:
                if DEVICE.type != 'cuda' or (sizes['queries'] == 1 and sizes['pairs'] == 1):
                    raise
            # 현재 스텝의 부분 gradient는 버리고, 같은 쿼리·오답과 RNG로 전체 스텝을 처음부터 다시 계산한다.
            optimizer.zero_grad(set_to_none=True)
            gc.collect()
            torch.cuda.empty_cache()
            restore_rng(rng)
            if sizes['queries'] > 1:
                sizes['queries'] = max(1, sizes['queries'] // 2)
            else:
                sizes['pairs'] = max(1, sizes['pairs'] // 2)
            log_line(
                log,
                f'OOM retry | micro queries {sizes["queries"]} | pairs {sizes["pairs"]} | '
                f'effective queries {EFFECTIVE_QUERIES} | tokens {CROSS_MAX_LENGTH}',
                True,
            )
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        sums += values
        counts += seen
        detail = loss_report(values[0], values[1], stop - start, values[2:], seen)
        log_line(
            log,
            f'epoch {epoch} | batch {step}/{len(starts)} | queries {stop}/{len(rows)} | '
            f'rank {detail["ranking_loss"]:.6f} | aspect {detail["aspect_loss"]:.6f} | '
            f'pair {detail["pair_loss"]:.6f} | total {detail["total_loss"]:.6f}',
            step == 1 or step % 100 == 0 or step == len(starts),
        )
    return loss_report(sums[0], sums[1], len(rows), sums[2:], counts)


@torch.inference_mode()
def evaluate(model, candidates, ann_ranks, val_groups, epoch, log, sizes):
    """손실용 후보는 추가로 채점하되 최종 지표는 원래 ColBERT 상위 100개만 재정렬해 계산한다."""
    model.eval()
    records = []
    feature_sums = np.zeros(len(ASPECT_KEYS))
    observed = np.zeros(len(ASPECT_KEYS), dtype=np.int64)
    rank_sum = 0.0
    pair_sum = 0.0
    for i, row in enumerate(VAL):
        qid = row['query_id']
        text = REVIEW_TEXTS[qid]
        docs = candidates[i].tolist()
        group = val_groups[i].tolist()
        gold = group[0]
        scored = list(dict.fromkeys(docs + group))
        while True:
            try:
                logits = rank_scores(model, [text] * len(scored), scored, sizes['eval_pairs']).cpu()
                break
            except torch.cuda.OutOfMemoryError:
                if DEVICE.type != 'cuda' or sizes['eval_pairs'] == 1:
                    raise
            gc.collect()
            torch.cuda.empty_cache()
            sizes['eval_pairs'] = max(1, sizes['eval_pairs'] // 2)
            log_line(log, f'VAL OOM retry | pairs {sizes["eval_pairs"]} | tokens {CROSS_MAX_LENGTH}', True)
        pos = {d: j for j, d in enumerate(scored)}
        selected = logits[[pos[d] for d in group]].unsqueeze(0)
        rank_loss = F.cross_entropy(selected, torch.zeros(1, dtype=torch.long)).item()
        pair_loss = pair_loss_per_query(selected).item()
        target = [TARGETS[qid]]
        attr = aspect_loss_sums(model, [text], target).cpu().numpy()
        seen = aspect_counts(target)
        feature_sums += attr
        observed += seen
        rank_sum += rank_loss
        pair_sum += pair_loss
        order = np.argsort(-logits[: len(docs)].numpy(), kind='stable')
        final = [docs[j] for j in order]
        rank = final.index(gold) + 1 if gold in docs else None
        records.append(
            {
                'epoch': epoch,
                'query_id': qid,
                'gold_item_id': row['item_id'],
                'gold_ann_rank_in_pool': int(ann_ranks[i]) or None,
                'gold_maxsim_rank_in_topk': docs.index(gold) + 1 if rank is not None else None,
                'cross_gold_rank': rank,
                'cross_top1_item_id': ITEM_IDS[final[0]],
                'cross_top10_item_ids': json.dumps([ITEM_IDS[d] for d in final[:10]], ensure_ascii=False),
                'cross_top100_item_ids': json.dumps([ITEM_IDS[d] for d in final], ensure_ascii=False),
                'ann_hit': bool(ann_ranks[i]),
                'colbert_hit': rank is not None,
                'cross_top1_hit': rank == 1,
                'cross_top10_hit': rank is not None and rank <= 10,
                'reciprocal_rank_at_100': 1.0 / rank if rank is not None else 0.0,
                'val_ranking_loss': rank_loss,
                'val_pair_loss': pair_loss,
                'val_aspect_loss': float(attr[seen > 0].mean()) if seen.any() else None,
            }
        )
        if (i + 1) % 50 == 0:
            log_line(log, f'epoch {epoch} | validation {i + 1}/{len(VAL)}')
    df = pd.DataFrame(records)
    loss = loss_report(rank_sum, pair_sum, len(VAL), feature_sums, observed)
    hit = int(df.colbert_hit.sum())
    summary = dict(
        model=MODEL_LABEL,
        epoch=epoch,
        n_queries=len(VAL),
        ann_recall_at_900=df.ann_hit.mean(),
        colbert_recall_at_100=df.colbert_hit.mean(),
        cross_precision_at_1=df.cross_top1_hit.mean(),
        cross_recall_at_10=df.cross_top10_hit.mean(),
        cross_mrr_at_100=df.reciprocal_rank_at_100.mean(),
        cross_top1_given_candidate=df.cross_top1_hit.sum() / hit if hit else None,
    )
    return df, loss, summary

## 8. 체크포인트와 결과 파일

| 새 출력 폴더 아래 경로 | 내용 |
|---|---|
| `checkpoints/<실험 ID>/epoch_01.pt` ~ `epoch_03.pt` | 전체 모델, AdamW 상태, epoch, config, 실험 ID와 난수 상태 |
| `logs/<실험 ID>/training_progress.log` | 후보 재사용과 BM25 단독 후보의 hard 4 진입 횟수, 학습 배치별 손실과 검증 상태 |
| `logs/<실험 ID>/input_length_diagnostics.csv` | 학습 전 길이·잘림 통계 |
| `logs/<실험 ID>/epoch_losses.csv` | TRAIN·VAL ranking, aspect, pair, total loss |
| `validation/<실험 ID>/epoch_XX/query_diagnostics.csv` | query_id, Gold 순위, 최종 Top-1·10·100, 쿼리별 검증 손실 |
| `validation/<실험 ID>/stage_summary.csv` | 제공된 DistilBERT epoch 3 기준점, ModernBERT 실측 지표와 차이 |
| `validation/<실험 ID>/best_epoch.json` | Recall@10 우선 최적 epoch와 체크포인트 경로 |

기준점은 사용자가 제공한 P@1 **0.2829**, R@10 **0.5257**, MRR@100 **0.3670**이다. 해당 모델을 다시 실행하지 않는다. ModernBERT 에포크끼리 Recall@10 동률이면 MRR@100, P@1, 이른 epoch 순서로 선택한다. Pair loss 설정과 세 가중치는 실험 ID에 포함된다. 이전 실험의 후보 캐시를 재사용할 수 있고, 새 가중치와 로그는 별도 실험 경로에 저장된다. 같은 설정으로 재실행하면 완료한 에포크를 복원하고, 검증 도중 끊긴 경우 그 체크포인트부터 평가한다.

In [ ]:
# 실험 ID에는 백본·토크나이저·데이터·검색·학습 조건을 넣어 다른 실험의 optimizer와 섞이지 않게 한다.
backend = json.loads(CROSS_TOKENIZER.backend_tokenizer.to_str())
backend.pop('padding', None)
backend.pop('truncation', None)
CONFIG = {
    'recipe_version': 3,
    'train_hard_mining': HARD_SPEC,
    'backbone': CROSS_BACKBONE,
    'revision': REVISION,
    'tokenizer_sha256': signature(backend),
    'max_length': CROSS_MAX_LENGTH,
    'pooling': 'masked_mean',
    'attention': 'sdpa',
    'retrieval': MINING_SPEC,
    'colbert_epoch': COLBERT_EPOCH,
    'aspect_vocabulary_sha256': sha256(VOCAB_PATH),
    'metadata_aspects_sha256': sha256(ASPECT_PATH),
    'ranking_weight': RANKING_LOSS_WEIGHT,
    'aspect_weight': ASPECT_LOSS_WEIGHT,
    'pair_weight': PAIR_LOSS_WEIGHT,
    'pair_objective': 'review_bundle_item_class_balanced_bce_with_logits',
    'negative_counts': {'hard': 4, 'random': 2, 'sibling': 1, 'same': 1},
    'epochs': EPOCHS,
    'seed': SEED,
    'lr': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'max_grad_norm': MAX_GRAD_NORM,
    'micro_queries': MICRO_BATCH_QUERIES,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'effective_queries': EFFECTIVE_QUERIES,
    'pair_micro_batch': PAIR_MICRO_BATCH,
    'eval_pair_batch': EVAL_PAIR_BATCH,
    'bf16': USE_BF16,
    'val_query_ids': [r['query_id'] for r in VAL],
    'torch': str(torch.__version__),
    'transformers': transformers.__version__,
}
del backend
MODEL_LABEL = 'ModernBERT + multi-aspect + pair BCE' if PAIR_LOSS_WEIGHT else 'ModernBERT + multi-aspect'
EXPERIMENT_ID = signature(CONFIG)
CKPT_DIR = OUTPUT_DIR / 'checkpoints' / EXPERIMENT_ID[:16]
VAL_DIR = OUTPUT_DIR / 'validation' / EXPERIMENT_ID[:16]
LOG_DIR = OUTPUT_DIR / 'logs' / EXPERIMENT_ID[:16]
for folder in (CKPT_DIR, VAL_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / 'training_progress.log'
print(
    '실험 ID',
    EXPERIMENT_ID[:16],
    '\n로그',
    LOG_PATH,
    '\n체크포인트',
    CKPT_DIR,
    '\n검증 결과',
    VAL_DIR,
    flush=True,
)


def atomic_write(path, write):
    """같은 파일시스템 안에서 완성된 체크포인트·표·설정만 최종 이름으로 공개한다."""
    with TemporaryDirectory(dir=path.parent) as tmp:
        staged = Path(tmp) / path.name
        write(staged)
        staged.replace(path)


def log_line(log, message, echo=False):
    """SageMaker 터미널에서도 진행을 따라갈 수 있도록 시각·메시지를 한 줄씩 flush한다."""
    line = f'{datetime.now().astimezone().isoformat(timespec="seconds")} | {message}'
    print(line, file=log, flush=True)
    if echo:
        print(line, flush=True)


def checkpoint_path(epoch):
    """완료한 ModernBERT epoch의 재개용 파일을 반환한다."""
    return CKPT_DIR / f'epoch_{epoch:02d}.pt'


def save_checkpoint(model, optimizer, epoch, metrics, sizes):
    """검증 전에 백본·7개 헤드, AdamW 모멘트, epoch, 난수와 학습 손실을 함께 보관한다."""
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'config': CONFIG,
        'experiment_id': EXPERIMENT_ID,
        'train_metrics': metrics,
        'rng': torch_rng(),
        'python_rng': random.getstate(),
        'batch_sizes': dict(sizes),
        'backbone_config': json.loads(model.encoder.config.to_json_string(use_diff=False)),
        'aspect_vocabularies': VOCABS,
    }
    atomic_write(checkpoint_path(epoch), lambda path: torch.save(state, path))


def restore_checkpoint(model, optimizer, epoch, sizes):
    """동일 실험의 완료 epoch만 허용하고 모델과 optimizer를 함께 복원한다."""
    path = checkpoint_path(epoch)
    state = load_pt(path)
    if state['experiment_id'] != EXPERIMENT_ID or state['config'] != CONFIG or state['epoch'] != epoch:
        raise ValueError(f'이 실험의 epoch {epoch} 체크포인트와 일치하지 않습니다. {path}')
    if not state['optimizer_state_dict']['state'] or any(
        g['lr'] != LEARNING_RATE for g in state['optimizer_state_dict']['param_groups']
    ):
        raise ValueError(f'AdamW step·학습률을 확인하세요. {path}')
    model.load_state_dict(state['model_state_dict'], strict=True)
    optimizer.load_state_dict(state['optimizer_state_dict'])
    restore_rng(state['rng'])
    random.setstate(state['python_rng'])
    sizes.update(state['batch_sizes'])
    return state['train_metrics']


def publish_validation(model, epoch, train_metrics, val_candidates, ann_ranks, groups, log, sizes):
    """쿼리 결과와 요약표를 함께 확정한다. 중단된 검증은 해당 epoch의 체크포인트부터 다시 평가한다."""
    folder = VAL_DIR / f'epoch_{epoch:02d}'
    if not folder.is_dir():
        log_line(log, f'epoch {epoch} | validation started', True)
        detail, val_metrics, summary = evaluate(model, val_candidates, ann_ranks, groups, epoch, log, sizes)
        summary.update({f'train_{k}': v for k, v in train_metrics.items()})
        summary.update({f'val_{k}': v for k, v in val_metrics.items()})
        summary.update(
            experiment_id=EXPERIMENT_ID,
            colbert_epoch=COLBERT_EPOCH,
            ann_nprobe=ANN_NPROBE,
            ann_per_token=ANN_PER_TOKEN,
            ann_pool=ANN_POOL,
            candidate_topk=CANDIDATE_TOPK,
            ranking_loss_weight=RANKING_LOSS_WEIGHT,
            aspect_loss_weight=ASPECT_LOSS_WEIGHT,
            pair_loss_weight=PAIR_LOSS_WEIGHT,
        )
        with TemporaryDirectory(dir=VAL_DIR) as tmp:
            staged = Path(tmp) / 'result'
            staged.mkdir()
            detail.to_csv(staged / 'query_diagnostics.csv', index=False)
            pd.DataFrame([summary]).to_csv(staged / 'stage_summary.csv', index=False)
            staged.rename(folder)
    result = pd.read_csv(folder / 'stage_summary.csv').iloc[0].to_dict()
    if (
        result['experiment_id'] != EXPERIMENT_ID
        or result['epoch'] != epoch
        or not (folder / 'query_diagnostics.csv').is_file()
    ):
        raise ValueError(f'저장된 검증 결과가 이번 epoch와 일치하지 않습니다. {folder}')
    return result


def write_comparison():
    """사용자 제공 DistilBERT 기준점과 실측 ModernBERT 결과를 구별하고 Recall@10 순서로 최적 epoch를 고른다."""
    # 재실행 초반에도 이미 확정된 뒤쪽 epoch를 표에서 잃지 않도록, 공개가 끝난 모든 보고서를 모은다.
    tables = []
    for epoch in range(1, EPOCHS + 1):
        folder = VAL_DIR / f'epoch_{epoch:02d}'
        if checkpoint_path(epoch).is_file() and (folder / 'query_diagnostics.csv').is_file():
            current = pd.read_csv(folder / 'stage_summary.csv')
            if (
                len(current) != 1
                or current.iloc[0]['experiment_id'] != EXPERIMENT_ID
                or current.iloc[0]['epoch'] != epoch
            ):
                raise ValueError(f'검증 요약과 현재 실험이 다릅니다. {folder}')
            tables.append(current)
    frame = pd.concat(tables, ignore_index=True)
    columns = [
        'epoch',
        'train_ranking_loss',
        'train_aspect_loss',
        'train_pair_loss',
        'train_total_loss',
        'val_ranking_loss',
        'val_aspect_loss',
        'val_pair_loss',
        'val_total_loss',
    ]
    atomic_write(LOG_DIR / 'epoch_losses.csv', lambda p: frame[columns].to_csv(p, index=False))
    reference = {
        'model': 'DistilBERT (provided baseline)',
        'epoch': 3,
        'colbert_epoch': 5,
        'n_queries': 350,
        'cross_precision_at_1': 0.2829,
        'cross_recall_at_10': 0.5257,
        'cross_mrr_at_100': 0.3670,
    }
    keys = ['cross_recall_at_10', 'cross_mrr_at_100', 'cross_precision_at_1', 'epoch']
    best = frame.sort_values(keys, ascending=[False, False, False, True]).iloc[0]
    summary = pd.concat([pd.DataFrame([reference]), frame], ignore_index=True)
    for k in ('cross_precision_at_1', 'cross_recall_at_10', 'cross_mrr_at_100'):
        summary[f'delta_{k}_vs_distilbert'] = summary[k] - reference[k]
    summary['selected_modernbert'] = summary.model.eq(MODEL_LABEL) & summary.epoch.eq(best.epoch)
    atomic_write(VAL_DIR / 'stage_summary.csv', lambda p: summary.to_csv(p, index=False))
    selected = {
        'epoch': int(best.epoch),
        'experiment_id': EXPERIMENT_ID,
        'checkpoint': str(checkpoint_path(int(best.epoch))),
        **{k: float(best[k]) for k in keys[:3]},
    }
    atomic_write(VAL_DIR / 'best_epoch.json', lambda p: write_json(p, selected))
    return summary


def run_experiment():
    """후보·길이 분포를 먼저 준비한 뒤 한 모델의 세 에포크를 순서대로 학습·검증한다."""
    global TARGETS
    import fcntl

    with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log:
        try:
            fcntl.flock(log, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError(f'다른 커널이 같은 실험을 실행 중입니다. {LOG_PATH}') from exc
        log_line(log, f'run started | {EXPERIMENT_ID[:16]} | BF16 {USE_BF16}', True)
        atomic_write(LOG_DIR / 'settings.json', lambda p: write_json(p, CONFIG))
        completed = [n for n in range(1, EPOCHS + 1) if checkpoint_path(n).is_file()]
        if completed != list(range(1, len(completed) + 1)):
            raise ValueError(f'중간 epoch 체크포인트가 없습니다. {CKPT_DIR}')
        TARGETS, coverage = init_targets()
        atomic_write(LOG_DIR / 'aspect_coverage.csv', lambda p: coverage.to_csv(p, index=False))
        print(coverage.to_string(index=False), flush=True)
        tables = candidate_tables(log)
        train_candidates = train_hard_candidates(TRAIN, tables['train'][0], log)
        val_candidates, ann_ranks = tables['val']
        val_groups = mixed_groups(VAL, val_candidates, random.Random(SEED))
        lengths_path = LOG_DIR / 'input_length_diagnostics.csv'
        if not lengths_path.is_file():
            log_line(log, '학습 전 입력 길이 측정 시작', True)
            rows, groups = epoch_groups(1, train_candidates)
            lengths = length_diagnostics(rows, groups, val_candidates)
            atomic_write(lengths_path, lambda p: lengths.to_csv(p, index=False))
            del rows, groups
        print(pd.read_csv(lengths_path).round(4).to_string(index=False), flush=True)
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        random.seed(SEED)
        torch.manual_seed(SEED)
        model = new_model(bool(completed))
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False
        )
        sizes = {'queries': MICRO_BATCH_QUERIES, 'pairs': PAIR_MICRO_BATCH, 'eval_pairs': EVAL_PAIR_BATCH}
        for number in range(1, EPOCHS + 1):
            if number in completed:
                train_metrics = restore_checkpoint(model, optimizer, number, sizes)
                log_line(log, f'epoch {number} | model + AdamW restored | {checkpoint_path(number)}', True)
            else:
                log_line(log, f'epoch {number} | training started | lr {LEARNING_RATE:g}', True)
                train_metrics = train_epoch(model, optimizer, number, train_candidates, log, sizes)
                save_checkpoint(model, optimizer, number, train_metrics, sizes)
                log_line(log, f'epoch {number} | checkpoint saved | {checkpoint_path(number)}', True)
            report = publish_validation(
                model, number, train_metrics, val_candidates, ann_ranks, val_groups, log, sizes
            )
            table = write_comparison()
            log_line(
                log,
                f'epoch {number} | train rank {report["train_ranking_loss"]:.5f} '
                f'aspect {report["train_aspect_loss"]:.5f} pair {report["train_pair_loss"]:.5f} '
                f'total {report["train_total_loss"]:.5f} | '
                f'val rank {report["val_ranking_loss"]:.5f} aspect {report["val_aspect_loss"]:.5f} '
                f'pair {report["val_pair_loss"]:.5f} total {report["val_total_loss"]:.5f} | '
                f'P@1 {report["cross_precision_at_1"]:.4f} '
                f'R@10 {report["cross_recall_at_10"]:.4f} MRR@100 {report["cross_mrr_at_100"]:.4f}',
                True,
            )
        return table

## 9. 실행

아래 셀이 후보 준비, 길이 분포 저장, 3개 에포크 학습과 VAL 평가를 수행한다. 파일 경로는 바로 위 셀에 출력된다. 별도 SageMaker 터미널에서는 `tail -n 30 -F <출력된 training_progress.log 경로>`로 진행을 볼 수 있다.

In [ ]:
results = run_experiment()
print(
    results[
        [
            'model',
            'epoch',
            'cross_precision_at_1',
            'cross_recall_at_10',
            'cross_mrr_at_100',
            'selected_modernbert',
        ]
    ]
    .round(4)
    .to_string(index=False)
)